In [1]:
# Experiment 6: Golden question set (30+ questions with expected answers and sources) + baseline run -> release v0.1
!pip -q install pymongo
import time, json, datetime
import pandas as pd
from google.colab import userdata
from google import genai
from pymongo import MongoClient

MODEL = "gemini-3.1-flash-lite"
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"), http_options={"timeout": 30000})
db = MongoClient(userdata.get("MONGODB_URI"), serverSelectionTimeoutMS=8000)["genai_course"]

# (question, expected keywords - all must appear, source in the course handout)
GOLDEN = [
 ("What does BPE stand for?", ["byte", "pair"], "Module I"),
 ("Which tokenizer does BERT use?", ["wordpiece"], "Module I"),
 ("Which tokenizer family do T5 and mT5 use?", ["sentencepiece"], "Module I"),
 ("Why does Telugu text usually cost more tokens than English?", ["token"], "Module I"),
 ("What is a knowledge cutoff of an LLM?", ["date"], "Module I"),
 ("What is a hallucination in an LLM?", ["false"], "Module I"),
 ("What is next-token prediction?", ["next", "token"], "Module I"),
 ("What is a context window?", ["token"], "Module I"),
 ("Name the three vectors used in attention.", ["query", "key", "value"], "Module II"),
 ("Which function turns attention scores into weights?", ["softmax"], "Module II"),
 ("Is GPT encoder-only or decoder-only?", ["decoder"], "Module II"),
 ("Is BERT encoder-only or decoder-only?", ["encoder"], "Module II"),
 ("What kind of architecture is T5?", ["encoder", "decoder"], "Module II"),
 ("What does a higher temperature do during sampling?", ["random"], "Module II"),
 ("What is top-k sampling?", ["k"], "Module II"),
 ("What is top-p sampling also called?", ["nucleus"], "Module II"),
 ("What does TF-IDF stand for?", ["term frequency", "inverse document frequency"], "Module III"),
 ("Name a classic sparse ranking function used in search engines.", ["bm25"], "Module III"),
 ("What does HNSW stand for?", ["hierarchical", "navigable", "small world"], "Module III"),
 ("What does RAG stand for?", ["retrieval", "augmented", "generation"], "Module III"),
 ("What does MRR stand for in retrieval evaluation?", ["mean", "reciprocal", "rank"], "Module III"),
 ("Why do we add overlap when chunking documents?", ["context"], "Module III"),
 ("What is few-shot prompting?", ["example"], "Module IV"),
 ("Which Python library is used for schema-validated structured outputs in this course?", ["pydantic"], "Module IV"),
 ("What is prompt injection?", ["instruction"], "Module IV"),
 ("What does LoRA stand for?", ["low", "rank", "adaptation"], "Module V"),
 ("What does the Q in QLoRA refer to?", ["quantiz"], "Module V"),
 ("What does DPO stand for?", ["direct", "preference", "optimization"], "Module V"),
 ("What does RLHF stand for?", ["reinforcement", "human", "feedback"], "Module V"),
 ("Which Python framework is used for the microservices in this course?", ["fastapi"], "Module VI"),
 ("Which tool is used for Infrastructure as Code on AWS in this course?", ["terraform"], "Module VI"),
 ("What does MCP stand for?", ["model", "context", "protocol"], "Module VI"),
]
golden = [{"id": f"Q{i+1:02d}", "question": q, "expected_keywords": k, "source": s} for i, (q, k, s) in enumerate(GOLDEN)]
json.dump(golden, open("golden_set.json", "w"), indent=2)
db.golden_set.delete_many({}); db.golden_set.insert_many([dict(g) for g in golden])
print("Golden set:", len(golden), "questions saved to golden_set.json and MongoDB (genai_course.golden_set)")

def ask(prompt, retries=4):
    for attempt in range(retries):          # retry with backoff on transient 5xx / timeouts
        try:
            return client.models.generate_content(model=MODEL, contents=prompt)
        except Exception as e:
            print("retry", attempt + 1, "->", str(e)[:60]); time.sleep(5 * (attempt + 1))
    raise RuntimeError("LLM call failed after retries")

def score(answer, keywords):
    a = answer.lower().replace("-", " ")
    return all(k.lower() in a for k in keywords)

results = []
for g in golden:
    t = time.time()
    r = ask("Answer in one or two sentences: " + g["question"])
    ans = (r.text or "").strip()
    results.append({"id": g["id"], "source": g["source"], "correct": score(ans, g["expected_keywords"]),
                    "latency_s": round(time.time() - t, 2), "input_tokens": r.usage_metadata.prompt_token_count,
                    "output_tokens": r.usage_metadata.candidates_token_count, "answer": ans[:90]})
    time.sleep(4)   # stay under free-tier requests-per-minute

df = pd.DataFrame(results)
print(df[["id", "source", "correct", "latency_s", "answer"]].to_string(index=False))
summary = {"release": "v0.1", "model": MODEL, "date": str(datetime.date.today()), "questions": len(df),
           "accuracy": round(df.correct.mean(), 3), "avg_latency_s": round(df.latency_s.mean(), 2),
           "avg_output_tokens": round(df.output_tokens.mean(), 1)}
db.releases.insert_one(dict(summary)); df.to_csv("baseline_v0.1.csv", index=False)
print("\nRelease v0.1 baseline:", summary)
print(df.groupby("source").correct.mean().round(2))

Golden set: 32 questions saved to golden_set.json and MongoDB (genai_course.golden_set)
 id     source  correct  latency_s                                                                                     answer
Q01   Module I     True       1.82 BPE stands for Byte Pair Encoding. It is a subword tokenization algorithm used in natural 
Q02   Module I     True       0.88 BERT uses the WordPiece tokenizer, which breaks words down into subword units based on the
Q03   Module I     True       1.25 T5 uses the **SentencePiece** tokenizer with the Unigram language model. Similarly, mT5 us
Q04   Module I     True       1.40 Telugu uses the Unicode script, which often requires multiple bytes or complex tokenizatio
Q05   Module I     True       0.85 A knowledge cutoff is the date marking the end of the data used to train an LLM, meaning t
Q06   Module I    False       1.68 A hallucination in an LLM occurs when the model generates information that is factually in
Q07   Module I     True       